# TC3-40 | Catálogo de Variáveis e Fontes de Dados Adicionais

**Responsável:** Sonia Andreia Ribeiro

**Sprint:** Fase 3

**Jira:** TC3-40

---

## Objetivo

Documentar as fontes de dados utilizadas no projeto, descrevendo suas variáveis, tipos de dados, chaves de integração e potencial de utilização na modelagem supervisionada.

Este catálogo servirá como referência para as atividades de Engenharia de Features (TC3-41), Avaliação de Data Leakage (TC3-42) e Pré-processamento (TC3-49).

In [7]:
import pandas as pd
from pathlib import Path

# 1. Fontes de Dados

O projeto utiliza uma Feature Table principal (`ml_aluno`) e duas bases adicionais previamente aprovadas pelo grupo para enriquecimento analítico.

As bases adicionais serão integradas em etapa posterior, preservando a rastreabilidade da camada Gold.

In [5]:
fontes = pd.DataFrame({
    "Fonte": [
        "ml_aluno",
        "Censo Escolar 2023",
        "INSE 2023"
    ],
    "Categoria": [
        "Camada Gold",
        "Educacional",
        "Socioeconômica"
    ],
    "Finalidade": [
        "Base principal para modelagem supervisionada",
        "Características das escolas e infraestrutura educacional",
        "Indicadores de nível socioeconômico dos estudantes"
    ]
})

fontes

,Fonte,Categoria,Finalidade
0,ml_aluno,Camada Gold,Base principal para modelagem supervisionada
1,Censo Escolar 2023,Educacional,Características das escolas e infraestrutura e...
2,INSE 2023,Socioeconômica,Indicadores de nível socioeconômico dos estuda...


# 2. Catálogo de Variáveis

Nesta etapa serão catalogadas as variáveis disponíveis em cada fonte de dados.

O objetivo é documentar a função de cada atributo, sem definir sua utilização na modelagem.

In [12]:
from pathlib import Path
import pandas as pd
import zipfile

# Caminho da base compactada
zip_path = Path("/content/sample_data/ml_aluno.zip")

# Pasta onde será extraída
extract_path = Path("/content/ml_aluno")

# Extrai apenas se ainda não existir
if not extract_path.exists():
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_path)

# Lê todos os arquivos parquet das subpastas
arquivos = sorted(extract_path.rglob("*.parquet"))

print(f"Arquivos encontrados: {len(arquivos)}")

df = pd.concat(
    [pd.read_parquet(arq) for arq in arquivos],
    ignore_index=True
)

print(f"Registros: {len(df):,}")
print(f"Variáveis: {df.shape[1]}")

df.head()

Arquivos encontrados: 69
Registros: 3,867,999
Variáveis: 19


,id_aluno,id_escola,id_municipio,nome_municipio,sigla_uf,ano_referencia,serie,rede,rede_descricao,caderno,presenca,preenchimento_caderno,proficiencia,peso_aluno,taxa_alfabetizacao_municipio_ano_anterior,media_portugues_municipio_ano_anterior,dataset_split,alfabetizado,processed_at
0,31062133,60021550,3136900,Juruaia,MG,2023,2,3,Municipal,6,1,1,731.235600,1.322718,NaN,NaN,treino,0,2026-09-01 18:03:53.794418
1,15078120,60001463,1506005,Prainha,PA,2023,2,3,Municipal,6,1,1,732.783236,1.324973,NaN,NaN,teste,0,2026-09-01 18:03:53.794418
2,43090743,60037407,4305108,Caxias do Sul,RS,2023,2,3,Municipal,6,1,1,755.341541,1.381868,NaN,NaN,treino,1,2026-09-01 18:03:53.794418
3,42097998,60035930,4219507,Xanxerê,SC,2023,2,3,Municipal,6,1,1,711.643712,2.012188,NaN,NaN,treino,0,2026-09-01 18:03:53.794418
4,43072338,60037775,4309209,Gravataí,RS,2023,2,3,Municipal,7,0,0,NaN,NaN,NaN,NaN,treino,0,2026-09-01 18:03:53.794418


In [22]:
# Recria o catálogo exatamente a partir do df
catalogo_gold = pd.DataFrame({
    "Variável": list(df.columns),
    "Tipo": df.dtypes.astype(str).tolist()
})

catalogo_gold["Fonte"] = "ml_aluno"
catalogo_gold["Descrição"] = ""
catalogo_gold["Categoria"] = ""

catalogo_gold

,Variável,Tipo,Fonte,Descrição,Categoria
0,id_aluno,object,ml_aluno,,
1,id_escola,object,ml_aluno,,
2,id_municipio,object,ml_aluno,,
3,nome_municipio,object,ml_aluno,,
4,sigla_uf,object,ml_aluno,,
5,ano_referencia,int32,ml_aluno,,
6,serie,int32,ml_aluno,,
7,rede,int32,ml_aluno,,
8,rede_descricao,object,ml_aluno,,
9,caderno,object,ml_aluno,,


# 2. Catálogo de Variáveis da Feature Table `ml_aluno`

Nesta etapa é gerado o dicionário de dados da Feature Table `ml_aluno`, identificando o nome de cada variável, seu tipo de dado, sua descrição funcional e sua categoria dentro do projeto.

Este catálogo tem como objetivo padronizar o entendimento das variáveis e servir como referência para as etapas de Engenharia de Features (TC3-41), Avaliação de Data Leakage (TC3-42) e Pré-processamento (TC3-49).

In [23]:
catalogo_gold["Descrição"] = catalogo_gold["Variável"].apply(
    lambda x: descricoes.get(x, ("Pendente", ""))[0]
)

catalogo_gold["Categoria"] = catalogo_gold["Variável"].apply(
    lambda x: descricoes.get(x, ("", "Revisar"))[1]
)

catalogo_gold

,Variável,Tipo,Fonte,Descrição,Categoria
0,id_aluno,object,ml_aluno,Identificador único do aluno,Identificador
1,id_escola,object,ml_aluno,Identificador único da escola,Identificador
2,id_municipio,object,ml_aluno,Código oficial do município (IBGE),Territorial
3,nome_municipio,object,ml_aluno,Nome do município,Territorial
4,sigla_uf,object,ml_aluno,Sigla da Unidade Federativa,Territorial
5,ano_referencia,int32,ml_aluno,Ano de referência da avaliação,Temporal
6,serie,int32,ml_aluno,Série escolar do estudante,Acadêmica
7,rede,int32,ml_aluno,Código da rede de ensino,Administrativa
8,rede_descricao,object,ml_aluno,Descrição da rede de ensino,Administrativa
9,caderno,object,ml_aluno,Código do caderno da avaliação,Avaliação


# 3. Catálogo de Variáveis — Censo Escolar 2023

O Censo Escolar 2023 é uma das bases adicionais aprovadas pelo grupo para enriquecimento da Feature Table `ml_aluno`.

Nesta seção será realizado o inventário das variáveis disponíveis, identificando seus tipos de dados e preparando o dicionário técnico que servirá de apoio para a Engenharia de Features (TC3-41).

In [28]:
from pathlib import Path

for arq in sorted(Path("/content").iterdir()):
    print(arq.name)

.config
INSE_2023_brasil.xlsx
INSE_2023_escolas.xlsx
INSE_2023_estados.xlsx
INSE_2023_municipios.xlsx
dicionário_dados_educação_básica.xlsx
microdados_ed_basica_2023.zip
ml_aluno
ml_aluno.zip
sample_data


In [30]:
from pathlib import Path
import pandas as pd
import zipfile

# Arquivo compactado do Censo
zip_censo = Path("/content/microdados_ed_basica_2023.zip")

# Pasta de extração
pasta_censo = Path("/content/censo_escolar_2023")

# Extrair apenas uma vez
if not pasta_censo.exists():
    with zipfile.ZipFile(zip_censo, "r") as zip_ref:
        zip_ref.extractall(pasta_censo)

print("Extração concluída!")

Extração concluída!


In [31]:
# Localizar todos os arquivos extraídos
for arq in sorted(pasta_censo.rglob("*")):
    if arq.is_file():
        print(arq)

/content/censo_escolar_2023/microdados_ed_basica_2023.csv


In [32]:
# Carregar a base do Censo Escolar 2023
df_censo = pd.read_csv(
    "/content/censo_escolar_2023/microdados_ed_basica_2023.csv",
    sep=";",
    encoding="latin1",
    low_memory=False
)

print(f"Registros: {len(df_censo):,}")
print(f"Variáveis: {df_censo.shape[1]}")

df_censo.head()

Registros: 217,625
Variáveis: 408


,NU_ANO_CENSO,NO_REGIAO,CO_REGIAO,NO_UF,SG_UF,CO_UF,NO_MUNICIPIO,CO_MUNICIPIO,NO_REGIAO_GEOG_INTERM,CO_REGIAO_GEOG_INTERM,...,QT_TUR_BAS_D,QT_TUR_BAS_N,QT_TUR_BAS_EAD,QT_TUR_INF_INT,QT_TUR_INF_CRE_INT,QT_TUR_INF_PRE_INT,QT_TUR_FUND_INT,QT_TUR_FUND_AI_INT,QT_TUR_FUND_AF_INT,QT_TUR_MED_INT
0,2023,Norte,1,Rondônia,RO,11,Porto Velho,1100205,Porto Velho,1101,...,14.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
1,2023,Norte,1,Rondônia,RO,11,Porto Velho,1100205,Porto Velho,1101,...,12.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2023,Norte,1,Rondônia,RO,11,Porto Velho,1100205,Porto Velho,1101,...,38.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2023,Norte,1,Rondônia,RO,11,Porto Velho,1100205,Porto Velho,1101,...,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2023,Norte,1,Rondônia,RO,11,Porto Velho,1100205,Porto Velho,1101,...,27.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 3.1 Inventário das Variáveis do Censo Escolar

O inventário abaixo identifica todas as variáveis presentes na base do Censo Escolar 2023, registrando seus tipos de dados. As descrições funcionais serão incorporadas a partir do dicionário oficial do INEP.

In [33]:
catalogo_censo = pd.DataFrame({
    "Variável": df_censo.columns,
    "Tipo": df_censo.dtypes.astype(str).values
})

catalogo_censo["Fonte"] = "Censo Escolar 2023"
catalogo_censo["Descrição"] = ""
catalogo_censo["Categoria"] = ""

print(f"Total de variáveis catalogadas: {len(catalogo_censo)}")

catalogo_censo.head(10)

Total de variáveis catalogadas: 408


,Variável,Tipo,Fonte,Descrição,Categoria
0,NU_ANO_CENSO,int64,Censo Escolar 2023,,
1,NO_REGIAO,object,Censo Escolar 2023,,
2,CO_REGIAO,int64,Censo Escolar 2023,,
3,NO_UF,object,Censo Escolar 2023,,
4,SG_UF,object,Censo Escolar 2023,,
5,CO_UF,int64,Censo Escolar 2023,,
6,NO_MUNICIPIO,object,Censo Escolar 2023,,
7,CO_MUNICIPIO,int64,Censo Escolar 2023,,
8,NO_REGIAO_GEOG_INTERM,object,Censo Escolar 2023,,
9,CO_REGIAO_GEOG_INTERM,int64,Censo Escolar 2023,,


## 3.2 Dicionário Oficial do Censo Escolar

O Censo Escolar disponibiliza um dicionário oficial contendo a descrição funcional de cada variável.

Nesta etapa, esse documento será utilizado para complementar automaticamente o catálogo técnico da base.

In [34]:
from pathlib import Path
import pandas as pd

arquivo_dic = Path("/content/dicionário_dados_educação_básica.xlsx")

xls = pd.ExcelFile(arquivo_dic)

print("Abas disponíveis:")
print(xls.sheet_names)

Abas disponíveis:
['microdados_unidade_coleta', 'suplemento_cursos_tecnicos']


## 3.3 Integração com o Dicionário Oficial do INEP

Para garantir consistência e rastreabilidade, as descrições das variáveis do Censo Escolar serão obtidas diretamente do dicionário oficial disponibilizado pelo INEP.

Esse processo evita documentação manual e mantém o catálogo reproduzível.

In [35]:
# Ler a aba principal do dicionário do INEP
dic_censo = pd.read_excel(
    arquivo_dic,
    sheet_name="microdados_unidade_coleta"
)

print(dic_censo.shape)
dic_censo.head()

(471, 24)


,Unnamed: 0,Alteração de nomenclatura ou metodologia de cálculo,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23
0,NaN,Variável nova,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,Descontinuidade,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Dicionário de Variáveis - Tabela de Escolas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [36]:
dic_preview = pd.read_excel(
    arquivo_dic,
    sheet_name="microdados_unidade_coleta",
    header=None
)

dic_preview.head(15)

,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
0,NaN,Alteração de nomenclatura ou metodologia de cá...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,Variável nova,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,Descontinuidade,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Dicionário de Variáveis - Tabela de Escolas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,N,Nome da Variável,Descrição da Variável,Tipo,Tam.(1),Categoria,"Coleta por ano (""s""=sim;""n""=não )",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Notas Importantes
7,NaN,NaN,NaN,NaN,NaN,NaN,07,08,09,10,...,15,16,17,18,19,20,21,22,23,NaN
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,1,NU_ANO_CENSO,Ano do Censo,Num,4,NaN,s,s,s,s,...,s,s,s,s,s,s,s,s,s,NaN


*Leitura estruturada do Dicionário do INEP*

Após inspecionar o arquivo, identificou-se que o cabeçalho do dicionário inicia na sétima linha da planilha. A leitura abaixo utiliza esse cabeçalho para estruturar corretamente os metadados das variáveis.

In [37]:
dic_censo = pd.read_excel(
    arquivo_dic,
    sheet_name="microdados_unidade_coleta",
    header=6
)

print(dic_censo.shape)
dic_censo.head()

(465, 24)


,N,Nome da Variável,Descrição da Variável,Tipo,Tam.(1),Categoria,"Coleta por ano (""s""=sim;""n""=não )",Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Notas Importantes
0,NaN,NaN,NaN,NaN,NaN,NaN,07,08,09,10,...,15,16,17,18,19,20,21,22,23,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,NU_ANO_CENSO,Ano do Censo,Num,4.0,NaN,s,s,s,s,...,s,s,s,s,s,s,s,s,s,NaN
3,2,NO_REGIAO,Nome da Região Geográfica,Char,20.0,NaN,s,s,s,s,...,s,s,s,s,s,s,s,s,s,NaN
4,3,CO_REGIAO,Código da Região Geográfica,Num,1.0,NaN,s,s,s,s,...,s,s,s,s,s,s,s,s,s,NaN


In [39]:
catalogo_censo = pd.DataFrame({
    "Variável": df_censo.columns,
    "Tipo Base": df_censo.dtypes.astype(str).values
})

catalogo_censo = catalogo_censo.merge(
    dic_censo[["Nome da Variável", "Descrição da Variável", "Tipo", "Categoria"]],
    left_on="Variável",
    right_on="Nome da Variável",
    how="left"
)

catalogo_censo = catalogo_censo.rename(columns={
    "Tipo": "Tipo Pandas"
})

catalogo_censo = catalogo_censo.drop(columns="Nome da Variável")

catalogo_censo.head(10)

,Variável,Tipo Base,Descrição da Variável,Tipo Pandas,Categoria
0,NU_ANO_CENSO,int64,Ano do Censo,Num,NaN
1,NO_REGIAO,object,Nome da Região Geográfica,Char,NaN
2,CO_REGIAO,int64,Código da Região Geográfica,Num,NaN
3,NO_UF,object,Nome da Unidade da Federação,Char,NaN
4,SG_UF,object,Sigla da Unidade da Federação,Char,NaN
5,CO_UF,int64,Código da Unidade da Federação,Num,NaN
6,NO_MUNICIPIO,object,Nome do Município,Char,NaN
7,CO_MUNICIPIO,int64,Código do Município,Num,NaN
8,NO_REGIAO_GEOG_INTERM,object,Nome da Região Geográfica Intermediária,Char,NaN
9,CO_REGIAO_GEOG_INTERM,int64,Código da Região Geográfica Intermediária,Num,NaN


# 4. Catálogo de Variáveis — INSE 2023

O Indicador de Nível Socioeconômico (INSE) representa a segunda base adicional aprovada pelo grupo para enriquecimento da Feature Table `ml_aluno`.

Nesta seção será realizado o inventário das variáveis da base de escolas, identificando seus tipos de dados e preparando o catálogo técnico para futura integração.

### 4.1 Critério de seleção da base INSE

O INSE 2023 é disponibilizado pelo INEP em quatro níveis de agregação: Brasil, estados, municípios e escolas.

Para este projeto foi selecionada exclusivamente a base **INSE_2023_escolas**, pois a Feature Table `ml_aluno` possui identificação da escola (`id_escola`), permitindo a integração em nível escolar.

As versões agregadas (Brasil, estados e municípios) não serão utilizadas, uma vez que representam indicadores resumidos e não oferecem o mesmo nível de granularidade necessário para o enriquecimento dos registros individuais dos alunos.

In [40]:
# Leitura da base INSE - Escolas
df_inse = pd.read_excel("/content/INSE_2023_escolas.xlsx")

print(f"Registros: {len(df_inse):,}")
print(f"Variáveis: {df_inse.shape[1]}")

df_inse.head()

Registros: 69,756
Variáveis: 22


,NU_ANO_SAEB,CO_UF,SG_UF,NO_UF,CO_MUNICIPIO,NO_MUNICIPIO,ID_ESCOLA,NO_ESCOLA,TP_TIPO_REDE,TP_LOCALIZACAO,...,MEDIA_INSE,INSE_CLASSIFICACAO,PC_NIVEL_1,PC_NIVEL_2,PC_NIVEL_3,PC_NIVEL_4,PC_NIVEL_5,PC_NIVEL_6,PC_NIVEL_7,PC_NIVEL_8
0,2023,11,RO,Rondônia,1100015,Alta Floresta D'Oeste,11024372,EMEIEF ANA NERY,3,1,...,4.29,Nível III,0.0,50.00,NaN,25.00,25.00,NaN,NaN,NaN
1,2023,11,RO,Rondônia,1100015,Alta Floresta D'Oeste,11024666,EMEIEF BOA ESPERANCA,3,2,...,4.88,Nível IV,0.0,3.64,12.87,33.01,50.49,NaN,NaN,NaN
2,2023,11,RO,Rondônia,1100015,Alta Floresta D'Oeste,11024682,EEEFM EURIDICE LOPES PEDROSO,2,1,...,4.91,Nível IV,0.0,13.32,17.68,23.27,23.92,13.20,7.70,0.90
3,2023,11,RO,Rondônia,1100015,Alta Floresta D'Oeste,11024828,EMEIEF IZIDORO STEDILE,3,1,...,5.03,Nível V,0.0,4.15,10.43,45.85,18.72,10.43,10.43,NaN
4,2023,11,RO,Rondônia,1100015,Alta Floresta D'Oeste,11024968,EEEMTI JUSCELINO KUBITSCHEK DE OLIVEIRA,2,1,...,5.21,Nível V,0.0,6.07,10.47,24.41,24.04,16.09,17.35,1.56


## 4.2 Inventário das Variáveis

Após a leitura da base INSE 2023, é realizado o inventário automático das variáveis, registrando seus tipos de dados e preparando o catálogo técnico para a etapa de enriquecimento da Feature Table `ml_aluno`.

In [41]:
catalogo_inse = pd.DataFrame({
    "Variável": df_inse.columns,
    "Tipo Pandas": df_inse.dtypes.astype(str).values
})

catalogo_inse["Fonte"] = "INSE 2023"
catalogo_inse["Descrição"] = ""
catalogo_inse["Categoria"] = ""

catalogo_inse.head(10)

,Variável,Tipo Pandas,Fonte,Descrição,Categoria
0,NU_ANO_SAEB,int64,INSE 2023,,
1,CO_UF,int64,INSE 2023,,
2,SG_UF,object,INSE 2023,,
3,NO_UF,object,INSE 2023,,
4,CO_MUNICIPIO,int64,INSE 2023,,
5,NO_MUNICIPIO,object,INSE 2023,,
6,ID_ESCOLA,int64,INSE 2023,,
7,NO_ESCOLA,object,INSE 2023,,
8,TP_TIPO_REDE,int64,INSE 2023,,
9,TP_LOCALIZACAO,int64,INSE 2023,,


In [42]:
# Leitura da base INSE - Escolas
df_inse = pd.read_excel("/content/INSE_2023_escolas.xlsx")

print(f"Registros: {len(df_inse):,}")
print(f"Variáveis: {df_inse.shape[1]}")

df_inse.head()

Registros: 69,756
Variáveis: 22


,NU_ANO_SAEB,CO_UF,SG_UF,NO_UF,CO_MUNICIPIO,NO_MUNICIPIO,ID_ESCOLA,NO_ESCOLA,TP_TIPO_REDE,TP_LOCALIZACAO,...,MEDIA_INSE,INSE_CLASSIFICACAO,PC_NIVEL_1,PC_NIVEL_2,PC_NIVEL_3,PC_NIVEL_4,PC_NIVEL_5,PC_NIVEL_6,PC_NIVEL_7,PC_NIVEL_8
0,2023,11,RO,Rondônia,1100015,Alta Floresta D'Oeste,11024372,EMEIEF ANA NERY,3,1,...,4.29,Nível III,0.0,50.00,NaN,25.00,25.00,NaN,NaN,NaN
1,2023,11,RO,Rondônia,1100015,Alta Floresta D'Oeste,11024666,EMEIEF BOA ESPERANCA,3,2,...,4.88,Nível IV,0.0,3.64,12.87,33.01,50.49,NaN,NaN,NaN
2,2023,11,RO,Rondônia,1100015,Alta Floresta D'Oeste,11024682,EEEFM EURIDICE LOPES PEDROSO,2,1,...,4.91,Nível IV,0.0,13.32,17.68,23.27,23.92,13.20,7.70,0.90
3,2023,11,RO,Rondônia,1100015,Alta Floresta D'Oeste,11024828,EMEIEF IZIDORO STEDILE,3,1,...,5.03,Nível V,0.0,4.15,10.43,45.85,18.72,10.43,10.43,NaN
4,2023,11,RO,Rondônia,1100015,Alta Floresta D'Oeste,11024968,EEEMTI JUSCELINO KUBITSCHEK DE OLIVEIRA,2,1,...,5.21,Nível V,0.0,6.07,10.47,24.41,24.04,16.09,17.35,1.56


### 4.3 Catálogo das Variáveis do INSE

O catálogo abaixo documenta as principais variáveis da base INSE 2023, registrando seu tipo de dado, finalidade e categoria funcional.

Este dicionário será utilizado como referência para a etapa de Engenharia de Features (TC3-41).

In [46]:
descricoes_inse = {
    "NU_ANO_SAEB": ("Ano de referência do SAEB", "Temporal"),
    "CO_UF": ("Código da Unidade Federativa", "Territorial"),
    "SG_UF": ("Sigla da Unidade Federativa", "Territorial"),
    "NO_UF": ("Nome da Unidade Federativa", "Territorial"),
    "CO_MUNICIPIO": ("Código oficial do município (IBGE)", "Territorial"),
    "NO_MUNICIPIO": ("Nome do município", "Territorial"),
    "ID_ESCOLA": ("Código identificador da escola", "Identificador"),
    "NO_ESCOLA": ("Nome da escola", "Identificador"),
    "TP_TIPO_REDE": ("Código do tipo de rede de ensino", "Administrativa"),
    "TP_LOCALIZACAO": ("Código da localização da escola", "Administrativa"),
    "MEDIA_INSE": ("Média do Indicador de Nível Socioeconômico", "Indicador Socioeconômico"),
    "INSE_CLASSIFICACAO": ("Classificação do nível socioeconômico", "Indicador Socioeconômico"),
    "PC_NIVEL_1": ("Percentual de estudantes no Nível I", "Distribuição INSE"),
    "PC_NIVEL_2": ("Percentual de estudantes no Nível II", "Distribuição INSE"),
    "PC_NIVEL_3": ("Percentual de estudantes no Nível III", "Distribuição INSE"),
    "PC_NIVEL_4": ("Percentual de estudantes no Nível IV", "Distribuição INSE"),
    "PC_NIVEL_5": ("Percentual de estudantes no Nível V", "Distribuição INSE"),
    "PC_NIVEL_6": ("Percentual de estudantes no Nível VI", "Distribuição INSE"),
    "PC_NIVEL_7": ("Percentual de estudantes no Nível VII", "Distribuição INSE"),
    "PC_NIVEL_8": ("Percentual de estudantes no Nível VIII", "Distribuição INSE"),
    "PC_NIVEL_9": ("Percentual de estudantes no Nível IX", "Distribuição INSE"),
    "PC_NIVEL_10": ("Percentual de estudantes no Nível X", "Distribuição INSE"),
    "TP_CAPITAL": (
        "Indicador se o município pertence a uma capital",
        "Territorial"
    ),
    "QT_ALUNOS_INSE": (
        "Quantidade de alunos considerados no cálculo do INSE",
        "Indicador Socioeconômico"
    )
}


catalogo_inse["Descrição"] = catalogo_inse["Variável"].map(
    lambda x: descricoes_inse.get(x, ("Pendente de documentação", "Revisar"))[0]
)

catalogo_inse["Categoria"] = catalogo_inse["Variável"].map(
    lambda x: descricoes_inse.get(x, ("", "Revisar"))[1]
)

catalogo_inse

,Variável,Tipo Pandas,Fonte,Descrição,Categoria
0,NU_ANO_SAEB,int64,INSE 2023,Ano de referência do SAEB,Temporal
1,CO_UF,int64,INSE 2023,Código da Unidade Federativa,Territorial
2,SG_UF,object,INSE 2023,Sigla da Unidade Federativa,Territorial
3,NO_UF,object,INSE 2023,Nome da Unidade Federativa,Territorial
4,CO_MUNICIPIO,int64,INSE 2023,Código oficial do município (IBGE),Territorial
5,NO_MUNICIPIO,object,INSE 2023,Nome do município,Territorial
6,ID_ESCOLA,int64,INSE 2023,Código identificador da escola,Identificador
7,NO_ESCOLA,object,INSE 2023,Nome da escola,Identificador
8,TP_TIPO_REDE,int64,INSE 2023,Código do tipo de rede de ensino,Administrativa
9,TP_LOCALIZACAO,int64,INSE 2023,Código da localização da escola,Administrativa


# 5. Chaves de Integração

As fontes de dados utilizadas no projeto apresentam diferentes nomenclaturas para identificar escolas e municípios.

Esta seção documenta as chaves que serão utilizadas posteriormente no Notebook de Enriquecimento, garantindo a rastreabilidade entre a Feature Table `ml_aluno` e as bases adicionais.

In [47]:
chaves_integracao = pd.DataFrame({
    "Base de Dados": [
        "ml_aluno",
        "Censo Escolar 2023",
        "INSE 2023"
    ],
    "Chave Escola": [
        "id_escola",
        "CO_ENTIDADE",
        "ID_ESCOLA"
    ],
    "Chave Município": [
        "id_municipio",
        "CO_MUNICIPIO",
        "CO_MUNICIPIO"
    ],
    "Granularidade": [
        "Aluno",
        "Escola",
        "Escola"
    ]
})

chaves_integracao

,Base de Dados,Chave Escola,Chave Município,Granularidade
0,ml_aluno,id_escola,id_municipio,Aluno
1,Censo Escolar 2023,CO_ENTIDADE,CO_MUNICIPIO,Escola
2,INSE 2023,ID_ESCOLA,CO_MUNICIPIO,Escola


# 6. Parecer Técnico

## Conclusão

O presente catálogo documentou as três fontes oficiais de dados utilizadas no projeto:

- **Feature Table `ml_aluno`**: base principal da modelagem supervisionada;
- **Censo Escolar 2023**: fonte complementar de informações educacionais e estruturais das escolas;
- **INSE 2023**: fonte complementar de indicadores de nível socioeconômico em nível escolar.

As variáveis foram catalogadas quanto ao tipo de dado, descrição funcional e categoria, estabelecendo a documentação necessária para as próximas etapas de Engenharia de Features (TC3-41), Avaliação de Data Leakage (TC3-42) e Pré-processamento (TC3-49).

Este notebook tem caráter documental e não realiza transformações ou integrações entre as bases, preservando a rastreabilidade dos dados originais.